# Operator Discovery And Validation\n\nThis notebook is the operator discovery and validation layer. It loads `trace_records.parquet`, normalizes trace behavior into production mining payloads, orchestrates operator mining through `src/offline/cluster_and_mine.py`, and writes an inspectable operator library plus diagnostics without inventing operators or silently filtering failures.\n

In [ ]:
from __future__ import annotations\n\nimport json\nimport math\nimport shutil\nfrom collections import Counter, defaultdict\nfrom itertools import combinations\nfrom pathlib import Path\n\nimport pandas as pd\n\nfrom src.common.schemas import OperatorLibraryEntry\nfrom src.offline.cluster_and_mine import (\n    MiningConfig,\n    cluster_and_mine,\n    operator_library_entries,\n    save_cluster_and_mine_artifacts,\n)\n\nSEED = 1337\nMAX_SINGLE_OPERATOR_SHARE = 0.70\nMIN_FAILURE_GAP_SUPPORT = 3\nUSAGE_CLUSTER_THRESHOLD = 0.50\nSTRUCTURAL_CLUSTER_THRESHOLD = 0.55\n\nROOT = Path.cwd()\nINTERIM_TRACE_PATH = ROOT / 'data' / 'interim' / 'trace_records.parquet'\nOFFLINE_TRACE_ROOT = ROOT / 'artifacts' / 'offline_trace_validation'\nRUN_ID = f'operator_mining_seed{SEED}'\nOUT_DIR = ROOT / 'artifacts' / 'operator_mining_validation' / RUN_ID\nOPERATOR_LIBRARY_PATH = OUT_DIR / 'operator_library.parquet'\nDIAGNOSTICS_PATH = OUT_DIR / 'operator_diagnostics.json'\nNORMALIZED_TRACE_PATH = OUT_DIR / 'normalized_trace_records.parquet'\n\n\ndef need(condition, message):\n    if not condition:\n        raise AssertionError(message)\n\n\ndef sj(value):\n    return json.dumps(value, ensure_ascii=True, sort_keys=True, default=str)\n\n\ndef text(value):\n    return ' '.join(str(value or '').split())\n\n\ndef is_missing(value):\n    return value is None or (isinstance(value, float) and math.isnan(value))\n\n\ndef parse_json(value, default):\n    if is_missing(value):\n        return default\n    if isinstance(value, type(default)):\n        return value\n    if isinstance(value, (list, dict)):\n        return value if isinstance(value, type(default)) else default\n    raw = str(value).strip()\n    if not raw:\n        return default\n    try:\n        parsed = json.loads(raw)\n    except Exception:\n        return default\n    return parsed if isinstance(parsed, type(default)) else default\n\n\ndef truthy(value):\n    if isinstance(value, bool):\n        return value\n    if isinstance(value, (int, float)) and not isinstance(value, bool):\n        return bool(value)\n    return text(value).lower() in {'1', 'true', 'yes', 'y', 'pass', 'passed'}\n\n\ndef fnum(value, default=0.0):\n    try:\n        return float(value)\n    except Exception:\n        return float(default)\n\n\ndef prepare_output_dir(path):\n    if path.exists():\n        shutil.rmtree(path)\n    path.mkdir(parents=True, exist_ok=True)\n\n\ndef resolve_trace_path():\n    if INTERIM_TRACE_PATH.exists():\n        return INTERIM_TRACE_PATH\n    candidates = sorted(OFFLINE_TRACE_ROOT.glob('**/trace_records.parquet'), key=lambda p: p.stat().st_mtime, reverse=True)\n    need(candidates, 'trace_records.parquet not found in data/interim or offline trace artifacts.')\n    return candidates[0]\n\n\ndef route_snapshot_from_row(row):\n    direct = parse_json(row.get('route_snapshot'), {})\n    if direct:\n        return direct\n    provenance = parse_json(row.get('provenance'), {})\n    route = provenance.get('route', {}) if isinstance(provenance, dict) else {}\n    return route if isinstance(route, dict) else {}\n\n\ndef step_list_from_row(row):\n    raw_steps = parse_json(row.get('steps'), []) or parse_json(row.get('step_sequence'), [])\n    if not isinstance(raw_steps, list):\n        raw_steps = []\n    out = []\n    for index, step in enumerate(raw_steps, start=1):\n        if not isinstance(step, dict):\n            continue\n        out.append(\n            {\n                'step_index': int(fnum(step.get('step_index', step.get('step_num', index)), index)),\n                'description': text(step.get('description') or step.get('text')),\n                'operator_used': text(step.get('operator_used') or step.get('operator_name')) or None,\n                'symbolic_valid': bool(step.get('symbolic_valid', True)),\n                'kind': text(step.get('kind')),\n                'phase': text(step.get('phase')),\n            }\n        )\n    return out\n\n\ndef operator_sequence_from_row(row, steps):\n    raw_sequence = parse_json(row.get('operator_sequence'), [])\n    sequence = [text(item) for item in raw_sequence if text(item)] if isinstance(raw_sequence, list) else []\n    if not sequence:\n        sequence = [step['operator_used'] for step in steps if step.get('operator_used')]\n    return sequence\n\n\ndef failure_labels_from_row(row):\n    labels = parse_json(row.get('failure_labels'), [])\n    out = [text(label) for label in labels if text(label)] if isinstance(labels, list) else []\n    for candidate in (row.get('failure_type'), row.get('failed_stage')):\n        clean = text(candidate)\n        if clean and clean not in out:\n            out.append(clean)\n    return out\n\n\ndef archetypes_from_row(row, route_snapshot):\n    raw_values = parse_json(row.get('archetypes'), [])\n    out = [text(item) for item in raw_values if text(item)] if isinstance(raw_values, list) else []\n    archetype_used = text(row.get('archetype_used'))\n    if archetype_used and archetype_used not in out:\n        out.append(archetype_used)\n    route_probs = route_snapshot.get('archetype_probs') or route_snapshot.get('archetypes') or {}\n    if isinstance(route_probs, dict):\n        ranked = sorted(route_probs.items(), key=lambda item: (-fnum(item[1]), str(item[0])))\n        for key, score in ranked:\n            clean = text(key)\n            if clean and fnum(score) > 0.0 and clean not in out:\n                out.append(clean)\n    return out\n\n\ndef row_to_raw_trace(row, index):\n    route_snapshot = route_snapshot_from_row(row)\n    steps = step_list_from_row(row)\n    operator_sequence = operator_sequence_from_row(row, steps)\n    failure_labels = failure_labels_from_row(row)\n    problem_id = text(row.get('problem_id') or row.get('id'))\n    branch_id = text(row.get('branch_id') or row.get('trace_id') or f'trace_{index:05d}')\n    need(problem_id, f'Row {index} is missing problem_id.')\n    answer = text(row.get('trace_answer') or row.get('answer')) or None\n    answer_canonical = text(row.get('trace_answer_canonical') or row.get('answer_canonical') or answer) or None\n    trace_status = text(row.get('trace_status') or row.get('record_type') or 'unknown')\n    symbolic_valid = truthy(row.get('symbolic_valid')) if 'symbolic_valid' in row else not failure_labels\n    return {\n        'record_type': 'raw_branch_trace',\n        'problem_id': problem_id,\n        'branch_id': branch_id,\n        'trace_id': branch_id,\n        'steps': steps,\n        'operator_sequence': operator_sequence,\n        'route_snapshot': route_snapshot,\n        'archetypes': archetypes_from_row(row, route_snapshot),\n        'answer': answer,\n        'answer_canonical': answer_canonical,\n        'symbolic_valid': symbolic_valid,\n        'verifier_score': fnum(row.get('verifier_score')),\n        'branch_score': fnum(row.get('branch_score')),\n        'tool_consistency': fnum(row.get('tool_consistency')),\n        'repaired': truthy(row.get('repaired')),\n        'repair_count': int(fnum(row.get('repair_count'))),\n        'retrieval_used': truthy(row.get('retrieval_used')),\n        'failure_type': failure_labels[0] if failure_labels else None,\n        'failure_labels': failure_labels,\n        'trace_status': trace_status,\n        'run_id': text(row.get('run_id')),\n        'problem_text': text(row.get('problem_text')),\n    }\n\n\ndef jaccard(left, right):\n    union = set(left) | set(right)\n    return 0.0 if not union else len(set(left) & set(right)) / len(union)\n\n\ndef cluster_feature_map(feature_map, threshold, prefix):\n    operators = sorted(feature_map)\n    parent = {name: name for name in operators}\n\n    def find(name):\n        while parent[name] != name:\n            parent[name] = parent[parent[name]]\n            name = parent[name]\n        return name\n\n    def union(left, right):\n        rl, rr = find(left), find(right)\n        if rl != rr:\n            parent[max(rl, rr)] = min(rl, rr)\n\n    for index, left in enumerate(operators):\n        for right in operators[index + 1:]:\n            if jaccard(feature_map[left], feature_map[right]) >= threshold:\n                union(left, right)\n\n    grouped = defaultdict(list)\n    for operator in operators:\n        grouped[find(operator)].append(operator)\n\n    ordered_groups = sorted((sorted(group) for group in grouped.values()), key=lambda group: (-len(group), group[0]))\n    operator_to_cluster = {}\n    cluster_rows = []\n    for index, group in enumerate(ordered_groups, start=1):\n        cluster_id = f'{prefix}_{index:03d}'\n        signature = sorted(set().union(*(feature_map[name] for name in group)))\n        cluster_rows.append({'cluster_id': cluster_id, 'members': group, 'size': len(group), 'signature': signature[:12]})\n        for name in group:\n            operator_to_cluster[name] = cluster_id\n    return operator_to_cluster, cluster_rows\n

In [ ]:
prepare_output_dir(OUT_DIR)\ntrace_path = resolve_trace_path()\ntrace_frame = pd.read_parquet(trace_path)\nneed(not trace_frame.empty, f'Trace source is empty: {trace_path}')\n\nraw_traces = []\nnormalized_rows = []\nfor index, row in enumerate(trace_frame.to_dict(orient='records')):\n    raw_trace = row_to_raw_trace(row, index)\n    raw_traces.append(raw_trace)\n    normalized_rows.append(\n        {\n            'problem_id': raw_trace['problem_id'],\n            'branch_id': raw_trace['branch_id'],\n            'trace_status': raw_trace['trace_status'],\n            'operator_sequence': raw_trace['operator_sequence'],\n            'step_count': len(raw_trace['steps']),\n            'archetype_used': raw_trace['archetypes'][0] if raw_trace['archetypes'] else None,\n            'archetypes': raw_trace['archetypes'],\n            'failure_labels': raw_trace['failure_labels'],\n            'success': bool(raw_trace['operator_sequence']) and not raw_trace['failure_labels'] and raw_trace['trace_status'] != 'stage_failure',\n        }\n    )\n\nnormalized_trace_frame = pd.DataFrame(normalized_rows)\nneed(not normalized_trace_frame.empty, 'No normalized traces were constructed from trace_records.parquet.')\n\nconfig = MiningConfig(\n    min_cluster_support=2,\n    min_distinct_problem_support=1,\n    min_symbolic_success_rate=0.55,\n    min_mean_verifier_score=0.55,\n    min_operator_purity=0.60,\n    max_failure_rate=0.45,\n    sequence_prefix=4,\n    miner_min_support=3,\n    miner_min_symbolic_success_rate=0.55,\n    miner_min_mean_verifier_gain=0.02,\n)\n\nartifact = cluster_and_mine(raw_traces, config=config)\nwrite_result = save_cluster_and_mine_artifacts(artifact, OUT_DIR / 'cluster_and_mine')\ncandidates = list(artifact.operator_candidates)\nentries = list(operator_library_entries(artifact, promoted_only=False))\n\nneed(candidates, 'Operator mining emitted no candidates.')\nneed(len(entries) == len(candidates), 'OperatorLibraryEntry count must align with operator candidates.')\n\nobserved = defaultdict(lambda: {'attempts': 0, 'successes': 0, 'failures': 0, 'archetypes': Counter(), 'failures_by_type': Counter(), 'previous': Counter(), 'next': Counter()})\nsequence_patterns = Counter()\nco_occurrence_edges = Counter()\nfailure_clusters = Counter()\narchetype_conditioned = defaultdict(lambda: {'attempts': 0, 'successes': 0, 'operators': Counter()})\n\nfor trace in raw_traces:\n    sequence = list(trace['operator_sequence'])\n    archetype = trace['archetypes'][0] if trace['archetypes'] else 'unknown'\n    trace_success = bool(sequence) and not trace['failure_labels'] and trace['trace_status'] != 'stage_failure'\n    for size in (2, 3):\n        for start in range(max(0, len(sequence) - size + 1)):\n            pattern = ' -> '.join(sequence[start:start + size])\n            if pattern:\n                sequence_patterns[pattern] += 1\n    for left, right in combinations(sorted(set(sequence)), 2):\n        co_occurrence_edges[(left, right)] += 1\n    if trace['failure_labels']:\n        cluster_key = '|'.join([archetype] + trace['failure_labels'] + (sequence[:2] or ['<none>']))\n        failure_clusters[cluster_key] += 1\n    for index, operator in enumerate(sequence):\n        stats = observed[operator]\n        stats['attempts'] += 1\n        stats['successes'] += int(trace_success)\n        stats['failures'] += int(not trace_success)\n        stats['archetypes'][archetype] += 1\n        for label in trace['failure_labels']:\n            stats['failures_by_type'][label] += 1\n        if index > 0:\n            stats['previous'][sequence[index - 1]] += 1\n        if index + 1 < len(sequence):\n            stats['next'][sequence[index + 1]] += 1\n        slice_stats = archetype_conditioned[archetype]\n        slice_stats['attempts'] += 1\n        slice_stats['successes'] += int(trace_success)\n        slice_stats['operators'][operator] += 1\n\ncandidate_by_name = {candidate.operator_name: candidate for candidate in candidates}\ntotal_attempts = sum(stats['attempts'] for stats in observed.values())\nneed(total_attempts > 0, 'No operator attempts were observed in trace_records.parquet.')\ndominance_share = max(stats['attempts'] for stats in observed.values()) / total_attempts\nneed(dominance_share <= MAX_SINGLE_OPERATOR_SHARE, f'Degenerate operator dominance detected: share={dominance_share:.3f}.')\n\nfor candidate in candidates:\n    need(candidate.preconditions, f'{candidate.operator_name} is missing mined preconditions.')\n    need(candidate.postconditions, f'{candidate.operator_name} is missing mined postconditions.')\n    need(all(text(item.description or item.field_name) for item in candidate.preconditions), f'{candidate.operator_name} has invalid preconditions.')\n    need(all(text(item.description) for item in candidate.postconditions), f'{candidate.operator_name} has invalid postconditions.')\n    empirical_failures = observed[candidate.operator_name]['failures']\n    if empirical_failures > 0:\n        need(candidate.failure_modes or observed[candidate.operator_name]['failures_by_type'], f'{candidate.operator_name} has failures without explicit failure modes.')\n\nusage_features = {}\nstructural_features = {}\nfor name, stats in observed.items():\n    candidate = candidate_by_name[name]\n    usage_features[name] = {\n        *{f'arch:{key}' for key, _ in stats['archetypes'].most_common(3)},\n        *{f'prev:{key}' for key, _ in stats['previous'].most_common(2)},\n        *{f'next:{key}' for key, _ in stats['next'].most_common(2)},\n        *{f'fail:{key}' for key, _ in stats['failures_by_type'].most_common(2)},\n    } or {f'op:{name}'}\n    structural_features[name] = {\n        f'family:{candidate.family.value}',\n        *{f'pre:{item.kind.value}:{item.field_name}:{item.operator}' for item in candidate.preconditions},\n        *{f'post:{item.kind.value}' for item in candidate.postconditions},\n        *{f'fail:{item.value}' for item in candidate.failure_modes},\n        *{f'tool:{item.value}' for item in candidate.tool_capabilities},\n    }\n\nusage_cluster_map, usage_clusters = cluster_feature_map(usage_features, USAGE_CLUSTER_THRESHOLD, 'usage')\nstructural_cluster_map, structural_clusters = cluster_feature_map(structural_features, STRUCTURAL_CLUSTER_THRESHOLD, 'structural')\n\nlibrary_rows = []\nfor entry in entries:\n    candidate = candidate_by_name[entry.operator]\n    stats = observed[entry.operator]\n    attempts = stats['attempts']\n    failures = stats['failures']\n    library_rows.append(\n        {\n            **OperatorLibraryEntry.model_validate(entry).model_dump(mode='json'),\n            'family': candidate.family.value,\n            'support_count': candidate.support_count,\n            'promotion_safe': candidate.promotion_safe,\n            'mean_verifier_gain': candidate.mean_verifier_gain,\n            'symbolic_success_rate': candidate.symbolic_success_rate,\n            'frequency': attempts,\n            'failure_rate_empirical': failures / max(attempts, 1),\n            'usage_cluster_id': usage_cluster_map[entry.operator],\n            'structural_cluster_id': structural_cluster_map[entry.operator],\n            'archetype_stats': sj(dict(stats['archetypes'])),\n            'failure_mode_histogram': sj(dict(stats['failures_by_type'])),\n        }\n    )\n\noperator_library_frame = pd.DataFrame(library_rows).sort_values(['promotion_safe', 'support_count', 'success_rate'], ascending=[False, False, False], kind='stable').reset_index(drop=True)\noperator_library_frame.to_parquet(OPERATOR_LIBRARY_PATH, index=False)\nnormalized_trace_frame.to_parquet(NORMALIZED_TRACE_PATH, index=False)\n\npromoted_names = {candidate.operator_name for candidate in candidates if candidate.promotion_safe}\ntop_operators = operator_library_frame[['operator', 'support_count', 'success_rate', 'failure_rate_empirical', 'usage_cluster_id']].head(15).to_dict(orient='records')\nfailure_prone_operators = operator_library_frame[operator_library_frame['support_count'] >= MIN_FAILURE_GAP_SUPPORT].sort_values(['failure_rate_empirical', 'support_count'], ascending=[False, False], kind='stable')[['operator', 'support_count', 'failure_rate_empirical', 'failure_modes']].head(15).to_dict(orient='records')\n\nmissing_operator_gaps = []\nfor archetype, slice_stats in sorted(archetype_conditioned.items()):\n    if slice_stats['attempts'] < MIN_FAILURE_GAP_SUPPORT:\n        continue\n    supported = sorted(name for name in promoted_names if archetype in set(candidate_by_name[name].compatible_archetypes))\n    if not supported:\n        missing_operator_gaps.append(\n            {\n                'gap_type': 'archetype_without_promoted_operator',\n                'archetype': archetype,\n                'attempts': slice_stats['attempts'],\n                'success_rate': slice_stats['successes'] / max(slice_stats['attempts'], 1),\n                'observed_operators': slice_stats['operators'].most_common(5),\n            }\n        )\n\nfor candidate in candidates:\n    if candidate.support_count >= MIN_FAILURE_GAP_SUPPORT and not candidate.promotion_safe:\n        missing_operator_gaps.append(\n            {\n                'gap_type': 'observed_but_not_promoted',\n                'operator': candidate.operator_name,\n                'support_count': candidate.support_count,\n                'success_rate': candidate.symbolic_success_rate,\n                'failure_modes': [item.value for item in candidate.failure_modes],\n            }\n        )\n\ndiagnostics = {\n    'source_trace_path': str(trace_path),\n    'output_dir': str(OUT_DIR),\n    'artifact_report': artifact.operator_report.model_dump(mode='json'),\n    'artifact_metadata': artifact.metadata.model_dump(mode='json'),\n    'cluster_and_mine_artifacts': write_result.model_dump(mode='json'),\n    'top_operators': top_operators,\n    'failure_prone_operators': failure_prone_operators,\n    'missing_operator_gaps': missing_operator_gaps[:20],\n    'sequence_patterns': [{'pattern': pattern, 'count': count} for pattern, count in sequence_patterns.most_common(20)],\n    'co_occurrence_graph': [{'left': left, 'right': right, 'weight': weight} for (left, right), weight in co_occurrence_edges.most_common(30)],\n    'failure_clusters': [{'cluster_key': key, 'count': count} for key, count in failure_clusters.most_common(20)],\n    'usage_clusters': usage_clusters,\n    'structural_clusters': structural_clusters,\n    'degenerate_operator_share': dominance_share,\n}\n\nDIAGNOSTICS_PATH.write_text(json.dumps(diagnostics, indent=2, ensure_ascii=True, sort_keys=True), encoding='utf-8')\ndiagnostics\n

In [ ]:
diagnostics = json.loads(DIAGNOSTICS_PATH.read_text(encoding='utf-8'))\noperator_library_frame = pd.read_parquet(OPERATOR_LIBRARY_PATH)\n\nsummary = {\n    'operator_count': int(len(operator_library_frame)),\n    'promoted_operator_count': int(operator_library_frame['promotion_safe'].sum()),\n    'top_operator': diagnostics['top_operators'][0]['operator'] if diagnostics['top_operators'] else None,\n    'most_failure_prone_operator': diagnostics['failure_prone_operators'][0]['operator'] if diagnostics['failure_prone_operators'] else None,\n    'missing_gap_count': len(diagnostics['missing_operator_gaps']),\n}\n\nprint(json.dumps(summary, indent=2, ensure_ascii=True, sort_keys=True))\noperator_library_frame.head(20)\n